# Head-to-Head Benchmark: From-Scratch (NumPy) vs. Industry Production Libraries (Scikit-Learn & XGBoost)
### *An Empirical Rigor & Performance Parity Study on 284,807 Financial Transactions*

**Author:** Manas Jain  
**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Objective:** Compare first-principles mathematical implementations against production-grade C++/Cython libraries (`scikit-learn`, `xgboost`) across:
1. **Classification Quality:** Accuracy, Precision, Recall, F1-Score, and ROC-AUC
2. **Computational Performance:** Training Latency (s) and Inference Latency (ms)
3. **Architectural Parity:** Evaluating where pure mathematical algorithms match or differ from enterprise packages.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Industry Libraries
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.tree import DecisionTreeClassifier as SklearnDecTree
from sklearn.ensemble import RandomForestClassifier as SklearnRandForest
from sklearn.neural_network import MLPClassifier as SklearnMLP
from sklearn.ensemble import IsolationForest as SklearnIsoForest
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
print("Libraries imported successfully.")

## 1. Data Ingestion & Preprocessing
Loading the official 284,807 transaction dataset and applying 50:50 balanced undersampling to mitigate the 0.17% class skew.

In [ ]:
DATA_PATH = os.path.join("..", "data", "creditcard.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("data", "creditcard.csv")

df = pd.read_csv(DATA_PATH)
print(f"Total Transactions: {len(df):,}")
print(f"Fraud Count: {df['Class'].sum():,} | Legit Count: {len(df) - df['Class'].sum():,}")

# Balanced 50:50 Undersampling
fraud_df = df[df['Class'] == 1]
legit_df = df[df['Class'] == 0]
legit_sample = legit_df.sample(n=len(fraud_df), random_state=SEED)
balanced = pd.concat([fraud_df, legit_sample], axis=0).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

feature_cols = [c for c in balanced.columns if c not in ['Time', 'Class']]
X_raw = balanced[feature_cols].to_numpy()
y_raw = balanced['Class'].to_numpy()

# 80/20 Partition
m = len(X_raw)
test_count = int(0.2 * m)
shuffled_idx = np.random.permutation(m)
train_idx, test_idx = shuffled_idx[test_count:], shuffled_idx[:test_count]

X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
y_train, y_test = y_raw[train_idx], y_raw[test_idx]

# Z-Score Normalization from scratch
mean = np.mean(X_train_raw, axis=0)
std = np.std(X_train_raw, axis=0)
std[std == 0.0] = 1.0

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

print(f"Train Split: {len(X_train)} samples | Test Split: {len(X_test)} samples")

## 2. Benchmark Evaluation Framework
Evaluating all models on the identical test split ($N=196$).

In [ ]:
def evaluate(y_true, y_pred, y_probs=None):
    TP = int(np.sum((y_pred == 1) & (y_true == 1)))
    FP = int(np.sum((y_pred == 1) & (y_true == 0)))
    TN = int(np.sum((y_pred == 0) & (y_true == 0)))
    FN = int(np.sum((y_pred == 0) & (y_true == 1)))
    total = TP + FP + TN + FN
    acc = (TP + TN) / total if total > 0 else 0.0
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    rec = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    roc_auc = 0.0
    if y_probs is not None:
        thresholds = np.linspace(0.0, 1.0, 100)
        tprs, fprs = [], []
        for t in thresholds:
            p = (y_probs >= t).astype(int)
            tp_t = np.sum((p == 1) & (y_true == 1))
            fp_t = np.sum((p == 1) & (y_true == 0))
            fn_t = np.sum((p == 0) & (y_true == 1))
            tn_t = np.sum((p == 0) & (y_true == 0))
            tprs.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0)
            fprs.append(fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0.0)
        s_idx = np.argsort(fprs)
        x_sort = np.array(fprs)[s_idx]
        y_sort = np.array(tprs)[s_idx]
        roc_auc = float(np.sum((y_sort[:-1] + y_sort[1:]) * 0.5 * np.diff(x_sort)))
        
    return {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'ROC-AUC': roc_auc
    }

## 3. Benchmark Results & Visual Insights
Load the pre-computed results and view the visual comparison figures.

In [ ]:
# Load pre-computed benchmark results
results_df = pd.read_csv("benchmark_results.csv")
results_df

In [ ]:
# Display Head-to-Head Figures
fig1_path = os.path.join("figures", "fig1_head_to_head_metrics.png")
fig2_path = os.path.join("figures", "fig2_head_to_head_speed.png")
fig3_path = os.path.join("figures", "fig3_roc_curves_comparison.png")

print(f"Figure 1: {fig1_path}")
print(f"Figure 2: {fig2_path}")
print(f"Figure 3: {fig3_path}")